Trying different type of models for LST estimation

## General paths, libraries and functions

In [ ]:
!pip install seaborn
!pip install statsmodels
!pip install rasterstats

In [ ]:
import os
import numpy as np
import rasterio
from rasterio import mask
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from rasterstats import zonal_stats
import joblib


In [ ]:
# user-defined paths
base_dir = '../../../datos-notebooks/3.global-data-cba/'
census_areas_name = ''
trained_models_path = '../../../datos-notebooks/1.HR-data-bologna/1.4.LST-prediction/' # modeles trained with HR data

In [ ]:
# inputs

# LST
lst_clipped_path = os.path.join(base_dir, 'LST/LST_clipped_celsius.tif') #values out of bound are nan
# vectors
census_areas_path = os.path.join(base_dir, census_areas_name)
# variables - Best results (pearson correlation with LST) for window size -- same as used in HR data
ndwi_1_path = os.path.join(base_dir, 'land-cover/moving-windows/ndwi/Class_1_window61.tif')
ndvi_1_path = os.path.join(base_dir, 'land-cover/moving-windows/ndvi/Class_1_window25.tif')
ndvi_0p5_path = os.path.join(base_dir, 'land-cover/moving-windows/ndvi/Class_0p5_window5.tif')
veg_vol_path = os.path.join(base_dir, 'objects-height/moving-windows/tree-volume/trees_window13.tif')
built_vol_path = os.path.join(base_dir, 'objects-height/moving-windows/building-volume/buildings_window13.tif')
built_area_path = os.path.join(base_dir, 'objects-height/moving-windows/building-area/building_area_window11.tif')
building_height_path = os.path.join(base_dir, 'objects-height/moving-windows/building-height/building_height_window21.tif')
mean_elev_path = os.path.join(base_dir, 'topography/moving_windows/elev_mean/elev_mean_win111.tif')
mean_slope_path = os.path.join(base_dir, 'topography/moving_windows/slope_mean/slope_mean_win19.tif')
northness_path = os.path.join(base_dir, 'topography/moving_windows/northness/northness_win201.tif')


In [ ]:
# output directorys
lst_prediction_folder = os.path.join(base_dir, 'LST-prediction/')
os.makedirs(lst_prediction_folder, exist_ok=True)
standarized_folder_path = os.path.join(lst_prediction_folder, 'standarized_variables')

## Dataset preparation

### mask rasters to city boundary

In [ ]:
with rasterio.open(lst_clipped_path) as src:
    lst = src.read(1)

lst_mask = np.isnan(lst)

# mask values out of city boundary
ndwi_1 = np.where(lst_mask, np.nan, rasterio.open(ndwi_1_path).read(1).astype("float32"))
ndvi_1 = np.where(lst_mask, np.nan, rasterio.open(ndvi_1_path).read(1).astype("float32"))
ndvi_0p5 = np.where(lst_mask, np.nan, rasterio.open(ndvi_0p5_path).read(1).astype("float32"))
veg_vol = np.where(lst_mask, np.nan, rasterio.open(veg_vol_path).read(1).astype("float32"))
built_vol = np.where(lst_mask, np.nan, rasterio.open(built_vol_path).read(1).astype("float32"))
built_area = np.where(lst_mask, np.nan, rasterio.open(built_area_path).read(1).astype("float32"))
built_height = np.where(lst_mask, np.nan, rasterio.open(building_height_path).read(1).astype("float32"))
mean_elev = np.where(lst_mask, np.nan, rasterio.open(mean_elev_path).read(1).astype("float32"))
mean_slope = np.where(lst_mask, np.nan, rasterio.open(mean_slope_path).read(1).astype("float32"))

# fixing "northness" for southern hemisphere
north = rasterio.open(northness_path).read(1).astype("float32") # Read northness raster
northness = np.where(lst_mask, np.nan, -north) # Invert northness (north -> south)

rasters_masked = {
    "Water": ndwi_1,
    "Dense vegetation presence": ndvi_1,
    "Sparse vegetation presence": ndvi_0p5,
    "Vegetation volume": veg_vol,
    "Built volume": built_vol,
    "Built area": built_area,
    "Building height": built_height,
    "Elevation": mean_elev,
    "Slope": mean_slope,
    "Northness": northness
}

In [ ]:
#plot masked rasters 

fig, axes = plt.subplots(4, 3, figsize=(20, 20))

for ax, (name, data) in zip(axes.flat, rasters_masked.items()):

    img = ax.imshow(data, cmap='viridis')
    ax.set_title(name)
    ax.axis('off')
    fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)


plt.tight_layout()
plt.show()

In [ ]:
# Plot histograms of masked rasters
fig, axes = plt.subplots(2, 5, figsize=(16, 8))
axes = axes.flatten()  # Flatten to 1D for easy indexing

for i, (name, arr) in enumerate(rasters_masked.items()):
    # Plot histogram using already-masked arrays
    axes[i].hist(arr[~np.isnan(arr)], bins=50)
    axes[i].set_title(name)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Frequency")

# Hide unused subplots (if less than 8 rasters)
for j in range(i + 1, 8):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

### standarize variables - mean:0, std:1

In [ ]:
os.makedirs(standarized_folder_path, exist_ok=True)

In [ ]:
# Z-score standardization for regression predictors

def standardize_array(arr, profile, out_path):
    mean = np.nanmean(arr)
    std = np.nanstd(arr)
    z = (arr - mean) / std

    profile.update(dtype='float32')

    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(z.astype('float32'), 1)

In [ ]:
# Apply standarization
with rasterio.open(lst_clipped_path) as src:
    profile = src.profile.copy()
    
standardize_array(ndwi_1, profile, os.path.join(standarized_folder_path, 'NDWI_1_z.tif'))
standardize_array(ndvi_1, profile, os.path.join(standarized_folder_path, 'NDVI_1_z.tif'))
standardize_array(ndvi_0p5, profile, os.path.join(standarized_folder_path, 'NDVI_0p5_z.tif'))
standardize_array(veg_vol, profile, os.path.join(standarized_folder_path, 'VEG_VOL_z.tif'))
standardize_array(built_vol, profile, os.path.join(standarized_folder_path, 'BUILT_VOL_z.tif'))
standardize_array(built_area, profile, os.path.join(standarized_folder_path, 'BUILT_AREA_z.tif'))
standardize_array(built_height, profile, os.path.join(standarized_folder_path, 'BUILDING_HEIGHT_z.tif'))
standardize_array(mean_elev, profile, os.path.join(standarized_folder_path, 'ELEV_z.tif'))
standardize_array(mean_slope, profile, os.path.join(standarized_folder_path, 'SLOPE_z.tif'))
standardize_array(northness, profile, os.path.join(standarized_folder_path, 'NORTH_z.tif'))

In [ ]:
std_rasters_paths = {
    "Water": os.path.join(standarized_folder_path, "NDWI_1_z.tif"),
    "Dense veg. presence": os.path.join(standarized_folder_path, "NDVI_1_z.tif"),
    "Sparse veg. presence": os.path.join(standarized_folder_path, "NDVI_0p5_z.tif"),
    "Vegetation Volume": os.path.join(standarized_folder_path, "VEG_VOL_z.tif"),
    #"Built Volume": os.path.join(standarized_folder_path, "BUILT_VOL_z.tif"),  # Excluded to prevent duplication of information
    "Built Area": os.path.join(standarized_folder_path, "BUILT_AREA_z.tif"),
    "Built Height": os.path.join(standarized_folder_path, "BUILDING_HEIGHT_z.tif"),
    "Elevation": os.path.join(standarized_folder_path, "ELEV_z.tif"),
    "Slope": os.path.join(standarized_folder_path, "SLOPE_z.tif"),
    "Northness": os.path.join(standarized_folder_path, "NORTH_z.tif")
}

In [ ]:
#plot cropped and standardized rasters

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for ax, (name, path) in zip(axes.flat, std_rasters_paths.items()):
    with rasterio.open(path) as src:
        data = src.read(1).astype(float)

    img = ax.imshow(data, cmap='viridis')
    ax.set_title(name)
    ax.axis('off')
    fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)


plt.tight_layout()
plt.show()

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 8))
axes = axes.flatten()  # Flatten to 1D for easy indexing

for i, (name, path) in enumerate(std_rasters_paths.items()):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        nodata = src.nodata

    # Plot histogram
    axes[i].hist(arr[~np.isnan(arr)], bins=50)
    axes[i].set_title(name)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Frequency")
    axes[i].axvline(0, linestyle='--')

plt.tight_layout()
plt.show()

### stratified sampling of LST

In [ ]:
rasters = {"LST": lst_clipped_path} # create dictionary with LST raster (not standardized)
rasters.update(std_rasters_paths) # add standardized rasters to the dictionary
#rasters.pop("Variable name", None) # to delete one variable if needed


In [ ]:
rasters

In [ ]:
# Create bins
n_bins = 10

# Read and flatten
data_arrays = {}
for name, path in rasters.items():
    with rasterio.open(path) as src:
        arr = src.read(1)
        data_arrays[name] = arr.flatten()

df = pd.DataFrame(data_arrays)

df = df.dropna() # Drop rows with NaN

df['LST_bin'] = pd.qcut(df['LST'], n_bins, labels=False) # Create LST bins for stratification 

In [ ]:
# Stratified sampling: 10% of each bin

sample_fraction = 0.1
df_sampled = df.groupby('LST_bin', group_keys=False).apply(
    lambda x: x.sample(frac=sample_fraction, random_state=42)
)

# Drop bin column
df_sampled = df_sampled.drop(columns=['LST_bin']).reset_index(drop=True)

print(df_sampled.head())


In [ ]:
# Show min, max and count of LST values in each bin

df_sampled['LST_bin'] = pd.qcut(df_sampled['LST'], n_bins, labels=False) # Recreate bins for the sampled dataset

for b in range(n_bins):
    values_in_bin = df_sampled[df_sampled['LST_bin'] == b]['LST']
    print(f"Bin {b}: min = {values_in_bin.min():.2f}, max = {values_in_bin.max():.2f}, count = {len(values_in_bin)}")

df_sampled = df_sampled.drop(columns=['LST_bin'])

### Descriptive statistics

In [ ]:
# Split features and target
X = df_sampled.drop(columns=['LST'])
y = df_sampled['LST']

In [ ]:
# Compute correlation matrix (only features)
corr_matrix = X.corr()

# Plot heatmap
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Feature Correlation Matrix")
plt.show()


In [ ]:
# Vegetation plot: area vs volume
plt.figure()
plt.scatter(
    df_sampled["Dense veg. presence"],
    df_sampled["Vegetation Volume"],
    s=1, alpha=0.5
)
plt.xlabel("Vegetation presence (z)")
plt.ylabel("Vegetation volume (z)")
plt.title("Vegetation: Area vs Volume")
plt.show()

# Built plot: area vs height
plt.figure()
plt.scatter(
    df_sampled["Built Area"],
    df_sampled["Built Height"],
    s=1, alpha=0.5
)
plt.xlabel("Built area (z)")
plt.ylabel("Built Height (z)")
plt.title("Built: Area vs Height")
plt.show()


In [ ]:
# Plot correlations: Variables vs LST

n_vars = len(X)

fig, axes = plt.subplots(2, 5, figsize=(10, 6))
axes = axes.flatten() 

for i, v in enumerate(X.columns):
    axes[i].scatter(X[v], y, s=1, alpha=0.3)
    axes[i].set_xlabel(v)
    axes[i].set_ylabel("LST")
    axes[i].set_title(f"LST vs {v}")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i, v in enumerate(df.columns[:10]):
    axes[i].boxplot(df[v], vert=False)
    axes[i].set_title(v)

for ax in axes[len(df.columns):]:
    ax.axis('off')

plt.tight_layout()
plt.show()


### PCA

In [ ]:
df_sampled

In [ ]:
y = df_sampled["LST"].values
X = df_sampled.drop(columns=["LST"])

In [ ]:
X

In [ ]:
# Load objects
scaler = joblib.load(os.path.join(trained_models_path, "scaler_pca.pkl"))
pca = joblib.load(os.path.join(trained_models_path, "pca.pkl"))
predictor_names = joblib.load(os.path.join(trained_models_path, "predictor_names_pca.pkl"))

# Apply to new data
X_std = scaler.transform(X)
X_pca = pca.transform(X_std)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(np.arange(1, len(pca.explained_variance_ratio_)+1),
         np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.ylim(0,1)
plt.title("Cumulative variance")
plt.grid(True)
plt.show()


In [ ]:
# Loadings
loadings = pd.DataFrame(
    pca.components_.T,       # each column is a PC
    columns=[f"PC{i+1}" for i in range(X.shape[1])],
    index=predictor_names     # original variable names
)

plt.figure(figsize=(12,6))
sns.heatmap(loadings, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("PC Loadings")
plt.show()


## Different models

### linear regression using PCs

In [ ]:
y = df_sampled["LST"].values
X_pca = pca.transform(X_std)

In [ ]:
# Load model
ols_pcs_model = joblib.load(os.path.join(trained_models_path, "ols_model_pcs.pkl"))

X_new = X_pca[:, :8] # Use already computed PCA

X_new_sm = sm.add_constant(X_new) # Add intercept
y_pred = ols_pcs_model.predict(X_new_sm) # Predict


In [ ]:
# Metrics 
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"R²: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")

### linear regression using PCs + stepwise

In [ ]:
import statsmodels.api as sm

# Load
ols_pcs_stepwise = joblib.load(os.path.join(trained_models_path, "ols_model_pcs_stepwise.pkl"))
selected_pcs = joblib.load(os.path.join(trained_models_path, "selected_pcs_stepwise.pkl"))

X_new = X_pca[:, :8]

# Select same PCs
pc_names = [f"PC{i+1}" for i in range(X_new.shape[1])]
final_idx = [pc_names.index(v) for v in selected_pcs]

X_new_sm = sm.add_constant(X_new)
X_new_final = X_new_sm[:, [0] + [i+1 for i in final_idx]]

# Predict
y_pred = ols_pcs_stepwise.predict(X_new_final)

In [ ]:
# Error metrics with test set
r2_test = r2_score(y, y_pred)
rmse_test = np.sqrt(mean_squared_error(y, y_pred))

print(f"\nTest set R²: {r2_test:.3f}")
print(f"Test set RMSE: {rmse_test:.3f}")


### linear regression using original variables

In [ ]:
X = df_sampled.drop(columns=['LST'])
y = df_sampled['LST']

In [ ]:
# Load model
ols_model = joblib.load(os.path.join(trained_models_path, "ols_model_variables.pkl"))

# Preprocess new data (same X definition)
X_new_sm = sm.add_constant(X)

# Predict
y_pred = ols_model.predict(X_new_sm)


In [ ]:
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"\nTest set R²: {r2:.3f}")
print(f"Test set RMSE: {rmse:.3f}")


### Linear regression using original variables + Stepwise

In [ ]:
X = df_sampled.drop(columns=['LST'])
y = df_sampled['LST']


In [ ]:
# Load
selected_features = joblib.load(
    os.path.join(trained_models_path, "selected_variables_stepwise.pkl")
)
ols_model = joblib.load(
    os.path.join(trained_models_path, "ols_model_variables_stepwise.pkl")
)

# Apply same feature selection
X = df_sampled[selected_features]

# Add intercept
X_new_const = sm.add_constant(X)

# Predict
y_pred = ols_model.predict(X_new_const)

In [ ]:
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"\nTest set R²: {r2:.3f}")
print(f"Test set RMSE: {rmse:.3f}")

### Linear regression using polynomial (grade 2) terms in original variables + stepwise

In [ ]:
X = df_sampled.drop(columns=['LST'])
y = df_sampled['LST']


In [ ]:
# Load
poly = joblib.load(os.path.join(trained_models_path, "poly.pkl"))
selected_poly_features = joblib.load(os.path.join(trained_models_path, "poly_stepwise.pkl"))
ols_poly_model = joblib.load(os.path.join(trained_models_path, "ols_model_poly_stepwise.pkl"))

# Apply same pipeline
X_new_poly = poly.transform(X)

poly_features = poly.get_feature_names_out(X.columns)
X_new_poly_df = pd.DataFrame(X_new_poly, columns=poly_features)

# Select same variables
X_new_sel = sm.add_constant(X_new_poly_df[selected_poly_features])

# Predict
y_pred = ols_poly_model.predict(X_new_sel)

In [ ]:
# Error metrics
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"\nTest set R²: {r2:.3f}")
print(f"Test set RMSE: {rmse:.3f}")

### Random Forest

In [ ]:
df_sampled

##### previously trained model

In [ ]:
# applying previously trained model 

X = df_sampled.drop(columns=['LST'])
y = df_sampled['LST']

#  Load previously trained RF model
rf_model = joblib.load(os.path.join(trained_models_path,"rf_model_bologna.pkl"))

# Predict
y_pred = rf_model.predict(X)

# Error metrics
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"R²: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")

In [ ]:
plt.figure()
plt.scatter(y, y_pred, s=1)  
plt.plot([y.min(), y.max()],
         [y.min(), y.max()], color="black")
plt.xlabel("Observed LST")
plt.ylabel("Predicted LST")
plt.title(f"Observed vs Predicted (R²={r2:.2f}, RMSE={rmse:.2f})")
plt.show()


In [ ]:
# Residuals plot
residuals = y - y_pred

plt.figure()
plt.scatter(y_pred, residuals, s=1)
plt.axhline(0, color='black')
plt.xlabel("Predicted LST")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted")
plt.show()


##### new model with this data

In [ ]:
X = df_sampled.drop(columns=['LST'])
y = df_sampled['LST']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Random Forest Regressor
rf_model = RandomForestRegressor(
    n_estimators=500,       # number of trees
    max_depth=None,         # no max depth
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)


In [ ]:
# Error metrics
y_pred = rf_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R²: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")




In [ ]:
# Error metrics -- checking with train data 
y_pred = rf_model.predict(X_train)
r2 = r2_score(y_train, y_pred)
rmse = np.sqrt(mean_squared_error(y_train, y_pred))

print(f"R²: {r2:.3f}") 
# it means 97% of variance is explained by the model in the training data -- from which 15% is overfitting (0.97-0.82=0.15)
print(f"RMSE: {rmse:.3f}")


In [ ]:
# Variable importance
importances = rf_model.feature_importances_

# Create dataframe for sorting
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": importances
})

# Sort by importance (descending)
importance_df = importance_df.sort_values(by="importance", ascending=False)

# Print ordered results
for _, row in importance_df.iterrows():
    print(f"{row['feature']}: {row['importance']:.3f}")

## Applying RF model in all the city

In [ ]:
print(X.columns.tolist())


In [ ]:
# Predictor order (MUST match training)
predictor_order = [
    "Water",
    "Dense veg. presence",
    "Sparse veg. presence",
    "Vegetation Volume",
    "Built Area",
    "Built Height",
    "Elevation",
    "Slope",
    "Northness"
]


# Read rasters using 'rasters' dict
arrays = []
meta = None

for name in predictor_order:
    path = rasters[name]   # get path from existing dict
    with rasterio.open(path) as src:
        arr = src.read(1)
        arrays.append(arr)
        if meta is None:
            meta = src.meta.copy()

stack = np.stack(arrays, axis=-1)  # (rows, cols, n_features)


# Mask nodata pixels
nodata_mask = np.any(np.isnan(stack), axis=-1)


# Reshape for model prediction
nrows, ncols, nbands = stack.shape
X_map = stack.reshape(-1, nbands)

valid = ~nodata_mask.ravel()
X_valid = X_map[valid]

# Predict LST
y_map = np.full(X_map.shape[0], np.nan)   # allocate output
y_map[valid] = rf_model.predict(X_valid) # calls the trained model

# Back to raster shape
lst_pred = y_map.reshape(nrows, ncols)

# Save raster
meta.update(dtype="float32", count=1, nodata=np.nan)

out_path = os.path.join(lst_prediction_folder, "LST_predicted_RF.tif")

with rasterio.open(out_path, "w", **meta) as dst:
    dst.write(lst_pred.astype("float32"), 1)
    

In [ ]:
# Plot observed vs predicted LST

lst_predicted_path = os.path.join(lst_prediction_folder, "LST_predicted_RF.tif")

with rasterio.open(lst_clipped_path) as src:
    lst_observed = src.read(1)

with rasterio.open(lst_predicted_path) as src:
    lst_pred = src.read(1)
    
vmin = np.nanmin(lst_observed)
vmax = np.nanmax(lst_observed)

# Plot 
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(lst_observed, vmin=vmin, vmax=vmax, cmap="hot")  
plt.colorbar(label="LST (°C)")
plt.title("Observed LST")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(lst_pred, vmin=vmin, vmax=vmax, cmap="hot")  
plt.colorbar(label="LST (°C)")
plt.title("Predicted LST")
plt.axis("off")

plt.tight_layout()
plt.show()
